In [1]:
from imports import *
from utils import *
from models import * # Should now contain UNet3D and AttentionUNet
from seed_everything import *
from Train_Eval_Test import *

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyperparameters adapted for 3D U-Net
args = {
    'device': device,
    'num_features': 1,      # Input channels (XCT grayscale)
    'f_maps': 32,           # Base filters for U-Net encoder
    'num_classes': 6,
    'lr': 0.001,
    'epochs': 200,
    'patience': 50,
    'classes': [0, 1, 2, 3, 4, 5]
}

### Setup of the dataset
We extract 3D patches ($64^3$) from the $512^3$ XCT volumes.

In [4]:
side = 512 
new_side = 64 
stride = 56 # Stride for training patch extraction

### Data Loading & Volumetric Reshaping
Instead of flattening voxels into graph nodes, we maintain the spatial 3D structure.

In [ ]:
# Load raw synthetic volumes
vols = [raw_to_tensor(f"FINAL{i}.raw", side) for i in range(1, 8)]
labs = [raw_to_tensor(f"LABELS{i}.raw", side) for i in range(1, 8)]

# Reshape into 5D patches: (N, Channel, D, H, W)
def extract_3d_patches(tensor, is_label=False):
    patches = view_as_windows(tensor.numpy(), (new_side, new_side, new_side), step=stride)
    if is_label:
        return torch.tensor(patches.reshape(-1, new_side, new_side, new_side))
    return torch.tensor(patches.reshape(-1, 1, new_side, new_side, new_side))

X_list = [extract_3d_patches(v) for v in vols]
Y_list = [extract_3d_patches(l, is_label=True) for l in labs]

X_total = torch.cat(X_list, dim=0)
Y_total = torch.cat(Y_list, dim=0)

### Class Weighting
Calculating inverse frequency weights to handle class imbalance (defects vs matrix).

In [ ]:
# Compute average occurrence across volumes for loss weighting
# (Simplified logic for demonstration)
inverse_norm_av = torch.ones(args['num_classes'])
inverse_norm_av[0] = 0 # Ignore voids per project requirements
np.save('weights_loss.npy', inverse_norm_av.numpy())

### Data Loaders
Using standard PyTorch DataLoader for 3D Tensors.

In [11]:
from torch.utils.data import DataLoader, TensorDataset, random_split

full_dataset = TensorDataset(X_total, Y_total)
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(4))

batch = 4
train_loader = DataLoader(train_data, batch_size=batch, shuffle=True, drop_last=True)
val_loader = DataLoader(val_data, batch_size=batch, shuffle=False)

### Training Loop (10 Runs with Early Stopping)

In [ ]:
save_dir = os.path.join(os.getcwd(), 'UNet_Models')
if not os.path.exists(save_dir): os.makedirs(save_dir)

for i in range(10):
    seed_everything(i)

    # Initialize 3D U-Net (or AttentionUNet)
    model = UNet3D(in_channels=args['num_features'], out_channels=args['num_classes'], f_maps=args['f_maps']).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'], weight_decay=0.001)
    loss_fn = torch.nn.CrossEntropyLoss(weight=inverse_norm_av.float().to(device))
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.96)

    best_loss = float('inf')
    patience_counter = args['patience']
    
    for epoch in range(1, args['epochs'] + 1):
        model.train()
        # train_function logic should now accept 5D tensors
        train_loss = train_step(model, train_loader, optimizer, loss_fn, device)
        
        model.eval()
        val_loss, val_dice = eval_step(model, val_loader, loss_fn, device)
        
        scheduler.step()
        
        print(f"Run {i} | Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Early Stopping
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, f'UNet_best_{i}.h5'))
            patience_counter = args['patience']
        else:
            patience_counter -= 1
            if patience_counter == 0:
                print("Early stopping triggered.")
                break